# 07 - Fine-tune ByT5 Arabic Diacritization (Glonor)

**Base Model:** `glonor/byt5-arabic-diacritization`
**Architecture:** ByT5 (300M params) - Byte-level Seq2Seq
**Current Score:** 0.47 (blind test)

## Why This Model?
- **Smaller & Faster**: 300M vs 720M (Fine-Tashkeel)
- **Byte-level**: Handles morphological variations well
- **Good baseline**: Already trained on Arabic diacritization

## Training Techniques:
1. **Curriculum Learning**: Progressive difficulty (short → long)
2. **Contrastive Learning**: Hard negatives from similar words
3. **Data Augmentation**: Word dropout, character swap
4. **Label Smoothing**: 0.1 for better generalization

## Tasks:
1. Load and preprocess KSAA training data
2. Fine-tune with curriculum learning
3. Evaluate on dev set
4. Generate test submission

In [1]:
!pip install -q transformers torch accelerate datasets tqdm pandas

In [2]:
# Setup
import os, sys, json, re, torch, gc, zipfile, random
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities (includes safeguards for empty responses & separated characters)
from utils_diacritization import (
    normalize_arabic, remove_diacritics, postprocess_diacritics,
    compute_metrics, sort_by_length, create_curriculum_batches, augment_sample,
    is_valid_output, repair_output, get_safe_seq2seq_generation_config
)

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'byt5_glonor_ft'

for d in [OUTPUT_DIR, SUBMISSION_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


## 1. Load Training Data with Curriculum Learning

In [3]:
train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples in TSV: {len(train_df)}")

def load_training_data():
    data = []
    text_dir = TRAIN_DIR / 'train_text'
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        txt_file = text_dir / f"{row['stem']}.txt"
        if txt_file.exists():
            with open(txt_file, 'r', encoding='utf-8') as f:
                diacritized = f.read().strip()
            
            diacritized = normalize_arabic(diacritized)
            undiacritized = remove_diacritics(diacritized)
            
            if len(undiacritized.split()) < 2:
                continue
                
            data.append({
                'id': row['stem'],
                'text_undiacritized': undiacritized,
                'text_diacritized': diacritized,
                'length': len(undiacritized)
            })
    return data

train_data = load_training_data()
print(f"Loaded {len(train_data)} training samples")

Training samples in TSV: 2327


Loading data: 100%|██████████| 2327/2327 [00:05<00:00, 395.39it/s]

Loaded 2317 training samples


In [4]:
# Create curriculum learning stages
NUM_CURRICULUM_STAGES = 3

# Sort by length
train_data_sorted = sort_by_length(train_data)

# Create stages (easy → medium → hard)
stage_size = len(train_data_sorted) // NUM_CURRICULUM_STAGES
curriculum_stages = [
    train_data_sorted[:stage_size],          # Easy: short sentences
    train_data_sorted[:2*stage_size],        # Medium: short + medium
    train_data_sorted                         # Hard: all sentences
]

for i, stage in enumerate(curriculum_stages):
    avg_len = sum(s['length'] for s in stage) / len(stage)
    print(f"Stage {i+1}: {len(stage)} samples, avg length: {avg_len:.0f} chars")

Stage 1: 772 samples, avg length: 30 chars
Stage 2: 1544 samples, avg length: 42 chars
Stage 3: 2317 samples, avg length: 59 chars


In [5]:
# Data augmentation
random.seed(42)

def augment_dataset(data, augment_prob=0.3):
    """Create augmented versions of the dataset."""
    augmented = []
    for sample in data:
        # Original
        augmented.append(sample)
        
        # Augmented version
        aug_in, aug_out = augment_sample(
            sample['text_undiacritized'],
            sample['text_diacritized'],
            augment_prob=augment_prob
        )
        if aug_in != sample['text_undiacritized']:
            augmented.append({
                'id': f"{sample['id']}_aug",
                'text_undiacritized': aug_in,
                'text_diacritized': aug_out,
                'length': len(aug_in)
            })
    return augmented

# Augment each curriculum stage
curriculum_stages_augmented = [
    augment_dataset(stage, augment_prob=0.3)
    for stage in curriculum_stages
]

for i, stage in enumerate(curriculum_stages_augmented):
    print(f"Stage {i+1} after augmentation: {len(stage)} samples")

Stage 1 after augmentation: 894 samples
Stage 2 after augmentation: 1853 samples
Stage 3 after augmentation: 2810 samples


## 2. Load Model

In [6]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset

MODEL_NAME = 'glonor/byt5-arabic-diacritization'
MODEL_KEY = 'byt5_glonor_ft'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(device)

model.gradient_checkpointing_enable()

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
clear_gpu()

Loading glonor/byt5-arabic-diacritization...


config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Model loaded: 299,637,760 parameters


## 3. Curriculum Learning Training

In [7]:
MAX_LENGTH = 512

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples['text_undiacritized'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        examples['text_diacritized'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Prepare eval dataset
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

dev_lookup = {item['id']: item['text_diacritized'] for item in dev_output}
dev_data = [
    {
        'text_undiacritized': normalize_arabic(item['text_undiacritized']),
        'text_diacritized': normalize_arabic(dev_lookup.get(item['id'], ''))
    }
    for item in dev_input if item['id'] in dev_lookup
]

eval_dataset = Dataset.from_list(dev_data)
eval_dataset = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['text_undiacritized', 'text_diacritized']
)

print(f"Eval dataset: {len(eval_dataset)} samples")

Map:   0%|          | 0/260 [00:00<?, ? examples/s]

Eval dataset: 260 samples


In [8]:
# Curriculum Learning: Train on progressively harder data - Optimized for 24GB GPU
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

EPOCHS_PER_STAGE = [2, 2, 3]  # More epochs on harder data
LEARNING_RATES = [5e-5, 3e-5, 2e-5]  # Lower LR as we progress

for stage_idx, (stage_data, epochs, lr) in enumerate(zip(
    curriculum_stages_augmented, EPOCHS_PER_STAGE, LEARNING_RATES
)):
    print(f"\n{'='*60}")
    print(f"CURRICULUM STAGE {stage_idx + 1}/{len(curriculum_stages_augmented)}")
    print(f"Samples: {len(stage_data)}, Epochs: {epochs}, LR: {lr}")
    print(f"{'='*60}")
    
    # Prepare dataset for this stage
    train_dataset = Dataset.from_list(stage_data)
    train_dataset = train_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=['id', 'text_undiacritized', 'text_diacritized', 'length']
    )
    
    # Training arguments for this stage - Optimized for 24GB GPU
    stage_checkpoint_dir = CHECKPOINTS_DIR / f'stage_{stage_idx + 1}'
    stage_checkpoint_dir.mkdir(exist_ok=True)
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(stage_checkpoint_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=8,   # Increased for 300M model
        per_device_eval_batch_size=16,   # Even larger for eval
        gradient_accumulation_steps=4,   # Effective batch: 32
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        label_smoothing_factor=0.1,
        fp16=True,  # FP16 for speed
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=50,
        report_to="none",
        predict_with_generate=True,
        generation_max_length=MAX_LENGTH,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
    )
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )
    
    clear_gpu()
    trainer.train()
    
    print(f"Stage {stage_idx + 1} complete!")
    clear_gpu()  # Clean between stages


CURRICULUM STAGE 1/3
Samples: 894, Epochs: 2, LR: 5e-05


Map:   0%|          | 0/894 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,No log,nan
2,123452.590000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,28.037722,nan
2,0.000000,nan
3,706.538437,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Stage 3 complete!


In [9]:
# Save final model
final_path = CHECKPOINTS_DIR / 'final'
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Model saved to: {final_path}")
clear_gpu()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./checkpoints/byt5_glonor_ft/final


## 4. Inference Functions

In [10]:
# Load fine-tuned model for inference (FP16 for speed)
final_model_path = CHECKPOINTS_DIR / 'final'
if final_model_path.exists():
    print(f"Loading fine-tuned model from {final_model_path}")
    # Clear training model first
    del model
    clear_gpu()
    
    model = AutoModelForSeq2SeqLM.from_pretrained(
        final_model_path,
        torch_dtype=torch.float16,  # FP16 for faster inference
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(final_model_path)
else:
    print("Using model from training")
    model = model.half()  # Convert to FP16

model.eval()
print("Model ready for inference (FP16)")

Loading fine-tuned model from ./checkpoints/byt5_glonor_ft/final


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model ready for inference (FP16)


In [11]:
@torch.inference_mode()
def diacritize(text: str, apply_postprocess: bool = True) -> str:
    """
    Diacritize Arabic text with safeguards against empty responses and separated characters.
    """
    original_text = text
    text = normalize_arabic(text)
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True
    ).to(device)
    
    # Get safe generation config
    gen_config = get_safe_seq2seq_generation_config()
    gen_config['max_length'] = MAX_LENGTH
    
    # Generate with safe parameters
    outputs = model.generate(
        **inputs,
        **gen_config
    )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Repair output (fixes separated characters, validates, falls back to input if invalid)
    if apply_postprocess:
        result = repair_output(result, original_text, fallback_to_input=True)
    
    return result

## 5. Evaluate on Dev Set

In [ ]:
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        result = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

Dev: 260 samples, 0 already done


Dev Set: 100%|██████████| 260/260 [09:07<00:00,  2.11s/it]


In [ ]:
# Compute metrics
print("\n" + "="*60)
print("DEV SET RESULTS")
print("="*60)

metrics = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab]")
print(f"  DER: {metrics['DER']*100:.2f}%")
print(f"  WER: {metrics['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics['SER']*100:.2f}%")

metrics_no_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=True)
print(f"\n[Excluding I'rab]")
print(f"  DER: {metrics_no_irab['DER']*100:.2f}%")


DEV SET RESULTS

[Including I'rab]
  DER: 28.19%
  WER: 70.34% (PRIMARY)
  SER: 99.62%

[Excluding I'rab]
  DER: 27.61%


## 6. Generate Test Submission

In [ ]:
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        result = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

Test: 328 samples, 0 already done


Test Set: 100%|██████████| 328/328 [10:39<00:00,  1.95s/it]


In [ ]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY")
print("="*60)
print(f"ZIP: {zip_file}")
print(f"Size: {zip_file.stat().st_size / 1024:.1f} KB")

SUBMISSION READY
ZIP: ./submissions/byt5_glonor_ft_submission_20260224_085535.zip
Size: 17.7 KB


In [7]:
# Sync (including checkpoints) and cleanup
import subprocess
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    print("Syncing to Google Drive (with checkpoints)...")
    subprocess.run(['bash', str(sync_script), '--with-checkpoints'])


# Final cleanup - free GPU memory
print("\nCleaning up GPU memory...")
del model
del tokenizer
clear_gpu()
print("Done! GPU memory released.")

Syncing to Google Drive (with checkpoints)...
KSAA 2026 - Sync Results to Google Drive
  Project: .

[1/3] Syncing outputs...
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                47 / 47, 100%
Elapsed time:         0.8sTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                50 / 50, 100%
Elapsed time:         1.1s
  Outputs synced to gKSAA:outputs

[2/3] Syncing submissions...
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                25 / 25, 100%
Elapsed time:         0.8sTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                27 / 27, 100%
Elapsed time:         0.9s
  Submissions synced to gKSAA:submissions

[3/3] Syncing checkpoints...
Transferred:   	          0 B / 24.326 GiB, 0%, 0 B/s, ETA -
Transferred:            0 / 147, 0%
Elapsed time:         1.1s
Transferring:
 *                    tashkeel_700m_ft/README.md:  0% /1.373Ki, 0/s, -
 *                 artst_v3_ft/final/config.json:  0% /2.034Ki,

NameError: name 'model' is not defined

In [6]:
# Google Drive sync removed for public release
